<a href="https://colab.research.google.com/github/villerbond/avito-text-orientation/blob/main/notebooks/03_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
!git clone https://github.com/villerbond/avito-text-orientation.git
%cd avito-text-orientation

Cloning into 'avito-text-orientation'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 97 (delta 8), reused 4 (delta 1), pack-reused 78 (from 1)
Receiving objects: 100% (97/97), 42.43 MiB | 20.00 MiB/s, done.
Resolving deltas: 100% (40/40), done.


In [70]:
PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent

In [80]:
import sys
import os
from pathlib import Path
import requests
import torch
import zipfile
from torch.utils.data import DataLoader, Dataset
from PIL import Image

from src.utils import set_seed, get_device
from src.data import build_test_loader
from src.models import build_model
from src.inference import predict

ImportError: cannot import name 'build_test_loader' from 'src.data' (/content/avito-text-orientation/src/data.py)

In [81]:
set_seed(42)
device = get_device()

In [82]:
TEST_DIR = PROJECT_DIR / "data" / "test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

In [83]:
YANDEX_URL = "https://disk.360.yandex.ru/d/ha0Q70T7ie-0_w"
ARCHIVE_PATH = PROJECT_DIR / "data" / "test_data.zip"

def download_yandex_file(public_url, save_path):
    api = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

    download_url = requests.get(
        api,
        params={"public_key": public_url}
    ).json()["href"]

    with requests.get(download_url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    print(f"Saved to {save_path}")


if not ARCHIVE_PATH.exists():
    download_yandex_file(YANDEX_URL, ARCHIVE_PATH)
else:
    print("Archive already exists.")

Saved to /content/avito-text-orientation/data/test_data.zip


In [84]:
if not any(TEST_DIR.iterdir()):
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as z:
        z.extractall(TEST_DIR)
    print("Archive extracted.")
else:
    print("Data already extracted.")

Archive extracted.


In [85]:
TEST_IMAGES_DIR = TEST_DIR / "test" / "images"

In [86]:
TEST_IMAGES_DIR

PosixPath('/content/avito-text-orientation/data/test/test/images')

In [87]:
BEST_MODEL_NAME = "resnet18"
CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / f"{BEST_MODEL_NAME}_best.pth"

In [88]:
model = build_model(BEST_MODEL_NAME).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

print(f"Модель {BEST_MODEL_NAME} загружена из {CHECKPOINT_PATH}")

Модель resnet18 загружена из /content/avito-text-orientation/checkpoints/resnet18_best.pth


In [89]:
test_loader = build_test_loader(TEST_IMAGES_DIR, batch_size=32)

In [ ]:
submission_df = predict(model, test_loader, device)

Inference:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
print(submission_df.shape)
submission_df.head()

(20000, 2)


,image_id,p_180
0,test_00000,0.003697
1,test_00001,0.999998
2,test_00002,0.999227
3,test_00003,0.006032
4,test_00004,0.997155


In [ ]:
PROJECT_DIR

In [ ]:
OUTPUT_PATH = PROJECT_DIR / "submission.csv"
submission_df.to_csv(OUTPUT_PATH, index=False)
print(f"Сохранено: {OUTPUT_PATH}")